In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Import Libraries

In [2]:
import pandas as pd
import numpy as np
import string

from transformers import AutoTokenizer, AutoModelForSequenceClassification,TrainingArguments, Trainer
from datasets import Dataset


from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
pip install sentence-transformers transformers datasets accelerate

Note: you may need to restart the kernel to use updated packages.


# EDA

In [4]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

print(train.shape)
print(test.shape)

(2000, 8)
(500, 7)


In [5]:
train.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [6]:
train['answer'].value_counts()

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

In [7]:
train.isnull().sum()

id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

# Evaluation Metric

In [8]:
def map3(actuals, predictions):
    score = 0

    for actual, pred in zip(actuals, predictions):
        if actual == pred[0]:
            score += 1
        elif actual == pred[1]:
            score += 1/2
        elif actual == pred[2]:
            score += 1/3

    return score / len(actuals)

In [9]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
Tesla T4


# Model 2 (Pretrained): DeBERTa

In [10]:
train_questions, val_questions = train_test_split(
    train,
    test_size=0.2,
    stratify=train["answer"],
    random_state=4524
)

print(train_questions.shape)
print(val_questions.shape)

(1600, 8)
(400, 8)


In [11]:
def expand_mcq(df):

    rows = []

    for _, row in df.iterrows():

        correct = row["answer"]

        for choice in ["A", "B", "C", "D", "E"]:

            rows.append({
                "prompt": row["prompt"],
                "option": row[choice],
                "label": int(choice == correct)
            })

    return pd.DataFrame(rows)

In [12]:
train_bert = expand_mcq(train_questions)
val_bert = expand_mcq(val_questions)

print(train_bert.shape)
print(val_bert.shape)

(8000, 3)
(2000, 3)


In [13]:
model_name = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [14]:
train_ds = Dataset.from_pandas(train_bert)
val_ds = Dataset.from_pandas(val_bert)

In [15]:
def tokenize(batch):

    return tokenizer(
        batch["prompt"],
        batch["option"],
        truncation=True,
        padding="max_length",
        max_length=384
    )

In [16]:
train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [17]:
train_ds = train_ds.remove_columns(
    ["prompt", "option"]
)

val_ds = val_ds.remove_columns(
    ["prompt", "option"]
)

train_ds = train_ds.rename_column(
    "label",
    "labels"
)

val_ds = val_ds.rename_column(
    "label",
    "labels"
)

In [18]:
train_ds.set_format(
    "torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]
)

val_ds.set_format(
    "torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]
)

In [19]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    torch_dtype=torch.float32
)

model = model.cuda()

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
classifier.bias         

In [20]:
training_args = TrainingArguments(
    output_dir="./deberta_base",
    learning_rate=1e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=False,
    bf16=False,
    report_to="none"
)

In [21]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds
)

In [22]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,No log,0.934294
2,0.916830,0.658970


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=500, training_loss=0.9168296508789062, metrics={'train_runtime': 328.75, 'train_samples_per_second': 48.669, 'train_steps_per_second': 1.521, 'total_flos': 1059776937984000.0, 'train_loss': 0.9168296508789062, 'epoch': 2.0})

In [23]:
def apk(actual, predicted, k=3):

    predicted = predicted[:k]

    for i, p in enumerate(predicted):

        if p == actual:
            return 1 / (i + 1)

    return 0


def mapk(actuals, predictions, k=3):

    return np.mean([
        apk(a, p, k)
        for a, p in zip(actuals, predictions)
    ])

In [24]:
choices = ["A", "B", "C", "D", "E"]

def predict_question(row):

    encodings = tokenizer(
        [row["prompt"]] * 5,
        [row[c] for c in choices],
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

    encodings = {
        k: v.to(model.device)
        for k, v in encodings.items()
    }

    with torch.no_grad():

        outputs = model(**encodings)

        probs = torch.softmax(
            outputs.logits,
            dim=1
        )[:, 1]

    probs = probs.cpu().numpy()

    order = np.argsort(probs)[::-1]

    return [choices[i] for i in order[:3]]

In [25]:
val_predictions = []

for _, row in val_questions.iterrows():

    val_predictions.append(
        predict_question(row)
    )

score = mapk(
    val_questions["answer"].tolist(),
    val_predictions
)

print("Validation MAP@3:", score)

Validation MAP@3: 0.89125


In [26]:
test_predictions = []

for _, row in test.iterrows():

    test_predictions.append(
        " ".join(
            predict_question(row)
        )
    )

# Submission Cell

In [27]:
submission = pd.DataFrame({
    "ID": test["id"],
    "Prediction": test_predictions
})

submission.to_csv("submission.csv", index=False)

submission.head()

,ID,Prediction
0,1,A C D
1,2,B D C
2,3,B E D
3,4,E C A
4,5,C A D
